# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aruu28249-boop/Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily performance of one content page for one pseudonymized client on one report date. For this project, I will use the `fact_content_daily_performance` table and work with a mid-panel month, March 2026 (`2026-03`), to avoid developing on the final-month sample. The decision is to use observed information available up to the decision point to prioritize content pages for human review or possible refresh. The final June 2026 month will be treated as a sealed future period rather than used for development.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

The five features I plan to use are:

- `gsc_impressions` — observed search impressions for the content item.
- `gsc_clicks` — observed search clicks for the content item.
- `gsc_avg_position` — observed average search position.
- `ga4_engaged_sessions` — observed engaged sessions when GA4 data is available.
- `scroll_events` — observed scroll events when engagement data is available.

These features describe observable search and engagement performance that may help prioritize content pages for human review.

### Label / Proxy

The target will be a future observed change in search clicks. A content page will be considered a potential refresh opportunity when its search clicks in a future evaluation period are lower than its search clicks during the current decision period. This future performance change will be used as a proxy for identifying pages that may need human review or content refresh. The future outcome will not be used as an input feature because doing so would cause target leakage.
### Context

The following fields provide context but are not direct predictive features:

- `content_hash_id` — identifies the pseudonymized content item and can be used for grouping or joining.
- `client_hash_id` — identifies the pseudonymized client and can be used for grouped analysis or splitting.
- `report_date` — identifies the date of the performance record.
- `month` — identifies the monthly partition used to select the March 2026 development slice.
- `client_has_gsc` — indicates whether GSC data is available for the client.
- `client_has_ga4` — indicates whether GA4 data is available for the client.

### Excluded

I will exclude `content_hash_id` and `client_hash_id` from model features because they are identifiers rather than meaningful performance signals. They may still be used for grouping, joining, or splitting the data.

I will also exclude `report_date` and `month` from the predictive feature set because they identify the observation period rather than describing the content's performance itself.

I will deliberately avoid using future-period outcome information as a feature because it would cause target leakage and make the evaluation unrealistically optimistic.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
import duckdb
from google.colab import userdata

# Create a DuckDB connection
con = duckdb.connect()

# Read Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to Hugging Face
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Warehouse location
rel = "hf://datasets/FlyRank/internship-warehouse"


# ============================================================
# Query 1: Verify the grain
# Expected grain:
# one row = one report_date + one client + one content item
# ============================================================

query1 = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

result1 = con.sql(query1)

print("Duplicate grain combinations:")
print(result1)


# ============================================================
# Query 2: Row count and date span for March 2026
# ============================================================

query2 = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

result2 = con.sql(query2)

print("March 2026 slice:")
print(result2)


# ============================================================
# Query 3: Check GSC data availability
# IS TRUE ensures only rows explicitly marked as available
# are counted.
# ============================================================

query3 = f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND client_has_gsc IS TRUE
"""

result3 = con.sql(query3)

print("Rows with GSC data available:")
print(result3)

Duplicate grain combinations:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────────┐
│ report_date │ client_hash_id │ content_hash_id │ row_count │
│    date     │    varchar     │     varchar     │   int64   │
├─────────────┴────────────────┴─────────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

March 2026 slice:
┌────────────┬───────────────┬─────────────┐
│ total_rows │ earliest_date │ latest_date │
│   int64    │     date      │    date     │
├────────────┼───────────────┼─────────────┤
│    9841378 │ 2026-03-01    │ 2026-03-31  │
└────────────┴───────────────┴─────────────┘

Rows with GSC data available:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│        9841378 │
└────────────────┘



In [18]:
# Five-feature frame for March 2026
# One row = one content item for one client on one report date

feature_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_engaged_sessions,
    scroll_events
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
LIMIT 10
"""

feature_df = con.sql(feature_query).df()

print("Feature frame shape:", feature_df.shape)
feature_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (10, 8)


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,<NA>,<NA>
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,7.832461,<NA>,<NA>
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,3.272727,<NA>,<NA>
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,5.636364,<NA>,<NA>
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,4.500000,<NA>,<NA>


### Five features and when they are available

- `gsc_impressions` — Available at the decision moment because it represents observed Google Search Console impressions recorded up to that date.
- `gsc_clicks` — Available at the decision moment because it represents observed Google Search Console clicks recorded up to that date.
- `gsc_avg_position` — Available at the decision moment because it summarizes the observed average search position up to that date.
- `ga4_engaged_sessions` — Available at the decision moment when GA4 data is available; missing values must be handled rather than treated as zero.
- `scroll_events` — Available at the decision moment when engagement data is available; missing values must be handled rather than treated as zero.

These features describe observed search and engagement performance at the time the review-prioritization decision is made. They should not include information from the future outcome period.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset can show observed patterns in content performance, but it cannot prove that a content refresh causes traffic, CTR, or ranking improvements. The warehouse is an unbalanced panel, so clients have different amounts of historical data, which limits direct comparisons across all clients. Some rows may also have missing or unavailable measurement data, so zero values should not always be interpreted as zero performance. Finally, overlapping time windows can create leakage if information from the future outcome period is used as a feature. Therefore, the results should be treated as directional and decision-support rather than causal proof.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.